In [0]:
import sys
from pyspark.sql import functions as F

sys.path.append("/Workspace/Shared")
import aws_utils

env = aws_utils.get_env_prefix()

business_date_param = dbutils.widgets.get("run_date").strip()

if business_date_param:
    run_date = business_date_param
else:
    run_date = (
        spark.table(f"`{env}-cdh`.derivative.dsettlementexposure")
        .filter(F.col("IsActiveFlag") == 1)
        .select(F.max("BusinessDate").alias("max_date"))
        .first()["max_date"]
    )
    run_date = str(run_date)


In [0]:
df_pa = (
    spark.table(f"`{env}-cdh`.derivative.dsettlementexposure")
    .filter(F.col("BusinessDate") == run_date)
    .filter(F.col("IsActiveFlag") == 1)
)

df_result = df_pa.select(
    F.col("SettlementKey").alias("settlement_code"),
    F.col("ExposureKey").alias("LoanBankCode"),
    F.col("ParticipationPercentage").alias("participation_perc"),
    F.col("SettlementExposureId").alias("SettlementId"),
    F.col("CaseCode").alias("case_code"),
    F.col("AddedManuallyFlag").cast("int").alias("added_manually_flg"),
    F.lit(1).alias("is_latest"),
    F.col("SettlementType").alias("settlement_type"),
    F.col("IsActiveFlag").alias("is_active"),
    F.col("IsMigratedFlag").cast("int").alias("is_migrated"),
    F.col("SubmissionDate").cast("date").alias("submission_dt"),
    F.col("ApprovalDate").alias("approval_dt"),
    F.col("ImplementationDate").cast("date").alias("implementation_dt"),
    F.col("CancelledDate").cast("date").alias("cancelled_dt"),
    F.col("RejectedDate").alias("rejected_dt"),
    F.col("SettlementStatusCode").cast("int").alias("status_id"),
    F.col("SettlementStatus").alias("StatusDescription"),
    F.col("LastUpdatedDate").cast("date").alias("status_change_date"),
    F.when(F.col("CalculatedSettlementStatus") == "Pending Implementation", 6230)
     .when(F.col("CalculatedSettlementStatus") == "Implemented", 6229)
     .when(F.col("CalculatedSettlementStatus") == "Fulfilled", 2027)
     .when(F.col("CalculatedSettlementStatus") == "Not Fulfilled", 2028)
     .when(F.col("CalculatedSettlementStatus") == "Partially Fulfilled", 2026)
     .cast("int").alias("eval_status_id"),
    F.col("CalculatedSettlementStatus").alias("EvalStatusDescription"),
    F.col("SettlementAmount").alias("settlement_amount"),
    F.col("DiscountAmount").alias("discount_amount"),
    F.col("ProposedAmount").alias("proposed_amount"),
    F.col("FirstInstalmentDate").alias("first_instalment_date"),
    F.col("ExpiredDate").cast("date").alias("expiry_dt"),
    F.col("NumOfinstalments").alias("num_of_instalments"),
    F.col("PastInstalmentAmount").alias("instalment_amount"),
    F.col("NextInstalmentDate").alias("next_instalment_date"),
    F.col("NextInstalmentAmount").alias("next_instalment_amount"),
    F.lit(None).cast("decimal(19,2)").alias("already_paid_amount"),
    F.col("RemainAmount").alias("remain_amount"),
    F.col("DebtAmount").alias("debt_amount"),
    F.col("TotalPaymentAmount").alias("tot_payment_amount"),
    F.col("ExpectedSettlementAmount").alias("expected_settle_amount"),
    F.col("Bucket").alias("settlement_bucket"),
    F.col("OverdueDate").alias("overdue_date"),
    F.col("PastInstalmentNumber").alias("past_instalments_num"),
    F.lit(None).cast("decimal(19,4)").alias("past_instalments_amount"),
    F.col("KeptPercentage").alias("kept_percentage"),
    F.col("CreatedDate").alias("created"),
    F.col("AutoCancelledDate").cast("date").alias("auto_cancelled_dt"),
    F.col("Comments").alias("comments"),
    F.col("MigrationComments").alias("migration_comments"),
    F.lit(None).cast("decimal(19,2)").alias("amount_calculated_as_paid"),
    F.col("PoolName").alias("pool_name"),
    F.col("NumOfaccounts").alias("num_of_accounts"),
    F.col("PrincipalProposedAmount").alias("principal_proposed_amount"),
)

In [0]:


(
    df_result.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"`{env}-cdh-delivery`.`it`.`CepalBILive_common_settlement`")
)